# Model Training — Smart Energy Optimizer
Trains and compares 3 models. Saves the best one as electricity_model.pkl

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model    import LinearRegression
from sklearn.ensemble        import RandomForestRegressor
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score
from xgboost                 import XGBRegressor

print('All libraries loaded.')

All libraries loaded.


## Step 1 — Load prepared dataset

In [2]:
df = pd.read_csv('../notebooks/daily_appliance_usage.csv')
# If running in same folder:
# df = pd.read_csv('daily_appliance_usage.csv')

print('Shape:', df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../notebooks/daily_appliance_usage.csv'

## Step 2 — Features and target split

In [ ]:
FEATURE_COLS = [
    'AC_Hours', 'AC_Tonnage',
    'Fridge_Hours', 'Fridge_Size',
    'Geyser_Hours',
    'WashingMachine_Hours',
    'Family_Members',
    'Season', 'Month', 'DayOfWeek',
    'Peak_Usage',
]

TARGET_COL = 'Total_kWh'  # Real global consumption — NOT derived from features

X = df[FEATURE_COLS]
y = df[TARGET_COL]

print('Features:', FEATURE_COLS)
print('Target:', TARGET_COL)
print('X shape:', X.shape, '| y shape:', y.shape)

## Step 3 — Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train rows:', len(X_train), '| Test rows:', len(X_test))

## Step 4 — Train and compare 3 models

This is required to justify your model choice to your supervisor.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    'XGBoost':           XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                      random_state=42, verbosity=0),
}

results = []
trained = {}

for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)

    mae  = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2   = r2_score(y_test, pred)

    results.append({'Model': name, 'MAE': round(mae, 4), 'RMSE': round(rmse, 4), 'R2': round(r2, 4)})
    trained[name] = m
    print(f'{name:25s}  MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}')

comparison = pd.DataFrame(results)
print('\n', comparison.to_string(index=False))

## Step 5 — Cross-validation on the best model

In [ ]:
# Pick best model by R2
best_name  = comparison.sort_values('R2', ascending=False).iloc[0]['Model']
best_model = trained[best_name]
print(f'Best model: {best_name}')

cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='r2')
print(f'Cross-validation R2 scores: {cv_scores.round(4)}')
print(f'Mean R2: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## Step 6 — Feature importance (required for FYP presentation)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    fi_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importances})
    fi_df = fi_df.sort_values('Importance', ascending=True)

    fi_df.plot.barh(x='Feature', y='Importance', figsize=(10, 6), color='steelblue', legend=False)
    plt.title(f'Feature importance — {best_name}')
    plt.xlabel('Importance score')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150)
    plt.show()
    print(fi_df.sort_values('Importance', ascending=False).to_string(index=False))
else:
    print('Linear Regression has coefficients instead of importances.')
    coef_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Coefficient': best_model.coef_})
    print(coef_df.to_string(index=False))

## Step 7 — HESCO bill calculator (correct slab rates)

In [ ]:
SLABS = [
    (50,          3.95),
    (100,         7.74),
    (200,        10.06),
    (300,        12.44),
    (700,        19.55),
    (float('inf'), 22.65),
]

def calculate_bill(monthly_units):
    monthly_units = round(monthly_units)
    energy = 0.0
    remaining  = monthly_units
    prev_limit = 0
    for limit, rate in SLABS:
        if remaining <= 0: break
        units_in = min(remaining, limit - prev_limit)
        energy  += units_in * rate
        remaining   -= units_in
        prev_limit   = limit
    gst          = energy * 0.18
    fc_surcharge = energy * 0.043
    total        = energy + gst + fc_surcharge + 400  # 400 = fixed charge
    return {
        'monthly_units':  monthly_units,
        'energy_charges': round(energy, 2),
        'gst':            round(gst, 2),
        'fc_surcharge':   round(fc_surcharge, 2),
        'fixed_charge':   400,
        'total_bill':     round(total, 2),
    }

# Test
for units in [100, 200, 350, 500, 750]:
    b = calculate_bill(units)
    print(f"{units:4d} units → Rs {b['total_bill']:,.0f}")

## Step 8 — Test prediction on a sample user

In [ ]:
sample_user = pd.DataFrame([{
    'AC_Hours':             8,
    'AC_Tonnage':           1.5,
    'Fridge_Hours':         24,
    'Fridge_Size':          2,
    'Geyser_Hours':         1,
    'WashingMachine_Hours': 0.5,
    'Family_Members':       5,
    'Season':               1,
    'Month':                7,
    'DayOfWeek':            3,
    'Peak_Usage':           1,
}])

daily_kwh   = best_model.predict(sample_user)[0]
monthly_kwh = daily_kwh * 30
bill        = calculate_bill(monthly_kwh)

print(f'Predicted daily kWh:   {daily_kwh:.3f}')
print(f'Estimated monthly kWh: {monthly_kwh:.1f}')
print(f'Estimated bill:        Rs {bill["total_bill"]:,.0f}')
print(f'  Energy charges:      Rs {bill["energy_charges"]:,.0f}')
print(f'  GST:                 Rs {bill["gst"]:,.0f}')
print(f'  FC surcharge:        Rs {bill["fc_surcharge"]:,.0f}')
print(f'  Fixed charge:        Rs {bill["fixed_charge"]:,.0f}')

## Step 9 — Save best model

In [ ]:
os.makedirs('../backend/model', exist_ok=True)
joblib.dump(best_model, '../backend/model/electricity_model.pkl')
print(f'Model saved: {best_name} → ../backend/model/electricity_model.pkl')

# Also save feature column names so the API knows the order
import json
with open('../backend/model/feature_cols.json', 'w') as f:
    json.dump(FEATURE_COLS, f)
print('Feature columns saved → ../backend/model/feature_cols.json')

In [ ]:
def train_anomaly_model(daily_df):
    """
    Train the Isolation Forest on the prepared daily dataset.
    Call this at the end of model_preparation.ipynb:

        from anomaly_detector import train_anomaly_model
        train_anomaly_model(ml_data)   # ml_data is your daily_appliance_usage df

    Saves anomaly_model.pkl and anomaly_scaler.pkl into the model/ folder.
    """
    os.makedirs('model', exist_ok=True)

    # Features that describe "how much energy a day uses"
    feature_cols = [
        'AC_Hours', 'Geyser_Hours', 'WashingMachine_Hours',
        'Family_Members', 'Season', 'Total_kWh'
    ]
    # Only keep columns that exist in this dataframe
    available = [c for c in feature_cols if c in daily_df.columns]
    X = daily_df[available].fillna(0).values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # contamination=0.05 → treat top 5% as anomalies
    iso = IsolationForest(
        n_estimators=200,
        contamination=0.05,
        random_state=42
    )
    iso.fit(X_scaled)

    joblib.dump(iso,    MODEL_PATH)
    joblib.dump(scaler, SCALER_PATH)

    print(f'[anomaly_detector] Trained on {len(X)} samples. Saved to model/')
    return iso, scaler



In [ ]:
from anomaly_detector import train_anomaly_model
os.chdir('../models')
iso_model, iso_scaler = train_anomaly_model(df)

# Verify: should flag ~5% as anomalous
from sklearn.preprocessing import StandardScaler
cols = [c for c in ['AC_Hours','Geyser_Hours','WashingMachine_Hours','Family_Members','Season','Total_kWh'] if c in df.columns]
X_iso = iso_scaler.transform(df[cols].fillna(0).values)
labels = iso_model.predict(X_iso)
pct = (labels==-1).mean()*100
print(f'Flags {pct:.1f}% as anomalous (target ~5%)')
os.chdir('../notebooks')